In [ ]:
import pandas as pd

# Carregar os dois arquivos
df_train = pd.read_csv("../data/processed/df_train_preprocessed.csv", index_col=0)

# Criar tabela de contagem por categoria
tabela_categorias = df_train["condition_label"].value_counts().reset_index()
tabela_categorias.columns = ["Categoria", "Quantidade"]

print(tabela_categorias)


In [ ]:
df_train.info()

In [ ]:
df_train['medical_abstract'] = df_train['medical_abstract'].str.lower()
df_train['medical_abstract'] = df_train['medical_abstract'].str.replace(r'\[.*?\]','', regex=True)
df_train['medical_abstract'] = df_train['medical_abstract'].str.replace(r'[^\w\s]','', regex=True)

display(df_train)

In [ ]:
def lower_replace(series):
    output = series.str.lower()
    output = output.str.replace(r'\[.*?\]','', regex=True)
    output = output.str.replace(r'[^\w\s]','', regex=True)

    return output

In [ ]:
lower_replace(df_train.medical_abstract)

In [ ]:
df = df_train.apply(lambda col: lower_replace(col) if col.dtype == "str" else col)
df.head(10)

In [ ]:
df.to_csv('../data/processed/data_process_pandas_1.csv')

In [ ]:
df_process = pd.read_csv('../data/processed/data_process_pandas_1.csv', index_col=0)
df_process

In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

# Função para normalizar texto
def lower_replace(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)       # remove conteúdo entre colchetes
    text = re.sub(r'[^\w\s]', '', text)       # remove pontuação
    return text

# Tokenização + lematização + remoção de stopwords
def token_lemma_stop(text: str) -> list:
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop]

# Filtrar apenas certas classes gramaticais (exemplo: substantivos e adjetivos)
def filter_pos(tokens: list) -> str:
    doc = nlp(" ".join(tokens))
    return " ".join([token.text for token in doc if token.pos_ in ["NOUN", "ADJ", "PRON", "VERB"]])


# Pipeline único
def preprocess(text: str) -> list:
    text = lower_replace(text)
    tokens = token_lemma_stop(text)
    return filter_pos(tokens)

# Aplicar no DataFrame
df_process['medical_abstract_clean'] = df_process['medical_abstract'].apply(preprocess)


In [ ]:
df_process.head(100)

In [ ]:
pd.to_pickle(df_process, '../data/processed/text_clean.pkl')

In [ ]:
text_clean = pd.read_pickle('../data/processed/text_clean.pkl')
text_clean

In [ ]:
# # Count Vectorizer

# text_clean = pd.read_pickle('../data/processed/text_clean.pkl')

# from sklearn.feature_extraction.text import CountVectorizer

# cv2 = CountVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.1, max_df=0.8)
# dtm2 = cv2.fit_transform(text_clean['medical_abstract_clean'])

# dtm2_df = pd.DataFrame(dtm2.toarray(), columns=cv2.get_feature_names_out())
# dtm2_df

In [ ]:
# term_freq = dtm2_df.sum()
# term_freq = term_freq.head(10)

# term_freq.sort_values().plot(kind='barh');

In [ ]:
# modelo de baseline TF_IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tv2 = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.2, max_df=0.8)
tfidf2 = tv2.fit_transform(text_clean.medical_abstract)
tfidf_df2 = pd.DataFrame(tfidf2.toarray(), columns=tv2.get_feature_names_out())
tfidf_df2

In [ ]:
# modelo de baseline TF_IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tv2 = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.2, max_df=0.8)
tfidf2 = tv2.fit_transform(text_clean.medical_abstract)
tfidf2
tfidf_df2 = pd.DataFrame(tfidf2.toarray(), columns=tv2.get_feature_names_out())
tfidf_df2

In [ ]:
# tfidf_df2.sum().sort_values().tail(10).plot(kind='barh')

In [ ]:
# tfidf_df2.sum().sort_values().head(10).plot(kind='barh')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd


# 2. Vetorização TF-IDF
tv = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.2, max_df=0.8)
X = tv.fit_transform(text_clean.medical_abstract)

# 3. Labels (escolha a coluna alvo)
y = text_clean['condition_name']  

# 4. Separar treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Treinar modelo Random Forest
rf_model = RandomForestClassifier(
    n_estimators=500, 
    max_depth=50, 
    random_state=42, 
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

# 6. Fazer predições
y_pred = rf_model.predict(X_test)

# 7. Avaliar
print(classification_report(y_test, y_pred))

In [ ]:
# Exemplo de novo texto
novo_texto = "i have a neoplasm problem"
# Pré-processar
# novo_texto_proc = preprocess(novo_texto)

# Vetorizar com o TF-IDF já treinado
X_novo = tv.transform([novo_texto_proc])
pred = rf_model.predict(X_novo)
print("Classe prevista:", pred[0])


In [ ]:
probs = rf_model.predict_proba(X_novo)
print("Probabilidades por classe:", probs)
